---
# **ALWAYS TESTING BEFORE PROCESS**
---

In [1]:
from selenium import webdriver

driver = webdriver.Chrome()
driver.get('http://pythonscraping.com')
driver.implicitly_wait(1)
print(driver.get_cookies())

[{'domain': '.pythonscraping.com', 'expiry': 1812771713, 'httpOnly': False, 'name': '_ga', 'path': '/', 'sameSite': 'Lax', 'secure': False, 'value': 'GA1.1.203839827.1778211713'}, {'domain': '.pythonscraping.com', 'expiry': 1812771713, 'httpOnly': False, 'name': '_ga_G60J5CGY1N', 'path': '/', 'sameSite': 'Lax', 'secure': False, 'value': 'GS2.1.s1778211713$o1$g0$t1778211713$j60$l0$h0'}]


---
# **MAIN PIPELINE**
---

In [4]:
import argparse
import csv
import gzip
import json
import logging
import random, re
import time
from dataclasses import asdict, dataclass, field
from datetime import datetime, timedelta
from io import BytesIO
from pathlib import Path
from typing import Optional
from urllib.parse import urlparse
import pandas as pd
 
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

# ── logging ────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("tiket_scraper.log", encoding="utf-8"),
    ],
)
log = logging.getLogger(__name__)
 
# ── constants ──────────────────────────────────────────────────────────────────
SITEMAP_INDEX_URL = "https://www.tiket.com/sitemap/id-id/index.xml.gz"
BASE_URL          = "https://www.tiket.com"
LOCALE            = "id-id"
PAGE_TIMEOUT      = 20
MAX_PROPERTIES    = 15
DRIVER_RECYCLE_EVERY = 15
 
# Child sitemap name patterns that contain property detail pages (PDP)
PDP_PATTERNS = {
    "hotel":     ["hotel-pdp"],
    "villa":     ["homes-villa"],
    "homes":     ["homes-pdp"],
    "glamping":  ["homes-glamping"],
    "cottage":   ["homes-cottage"],
    "apartment": ["homes-apartment"],
}

In [47]:
import gzip, os, time, glob, platform
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import xml.etree.ElementTree as ET

def wait_for_download(download_folder: str, timeout: int = 30) -> str:
    """Wait until a file finishes downloading, return its path."""
    print("Waiting for download to complete...")
    deadline = time.time() + timeout
    while time.time() < deadline:
        # .crdownload = Chrome's temp file while still downloading
        files = glob.glob(os.path.join(download_folder, "*.gz"))
        tmp   = glob.glob(os.path.join(download_folder, "*.crdownload"))
        if files and not tmp:
            latest = max(files, key=os.path.getmtime)
            print(f"Download complete → {latest}")
            return latest
        time.sleep(1)
    raise TimeoutError("Download did not complete in time")


def build_driver(download_folder: str, headless: bool = False) -> webdriver.Chrome:
    opts = Options()

    # ── redirect all downloads to project folder ───────────────────────────────
    prefs = {
        "download.default_directory":        download_folder,
        "download.prompt_for_download":      False,   # no "Save As" popup
        "download.directory_upgrade":        True,
        "safebrowsing.enabled":              True,
        "plugins.always_open_pdf_externally": True,
    }
    opts.add_experimental_option("prefs", prefs)

    # ── anti-bot ───────────────────────────────────────────────────────────────
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)

    # ── VPS / headless safe flags ──────────────────────────────────────────────
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")                  # required on VPS
    opts.add_argument("--disable-dev-shm-usage")       # prevents memory crash on VPS
    opts.add_argument("--disable-gpu")                 # no GPU on VPS
    opts.add_argument("--window-size=1280,900")
    opts.add_argument("--disable-extensions")

    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=opts)

    # ── allow downloads in headless mode (Chrome blocks by default) ────────────
    if headless:
        driver.execute_cdp_cmd(
            "Page.setDownloadBehavior",
            {"behavior": "allow", "downloadPath": download_folder},
        )

    return driver

def download_sitemap(url: str, download_folder: str, headless: bool = False) -> str:
    """
    Open Chrome, navigate to sitemap URL, let Chrome auto-download the .gz,
    wait for it, then return the local file path.
    """
    os.makedirs(download_folder, exist_ok=True)
    driver = build_driver(download_folder, headless=headless)

    try:
        # Visit homepage first so Cloudflare sets cookies
        print("Visiting homepage to warm up session...")
        driver.get("https://www.tiket.com")
        time.sleep(5)

        if not headless:
            # input(">>> Solve Cloudflare challenge if it appears, then press ENTER ...")
            time.sleep(10)

        # Trigger the download
        print(f"Downloading: {url}")
        driver.get(url)

        # Wait for Chrome to finish downloading
        gz_path = wait_for_download(download_folder)
        driver.quit()
        return gz_path

    finally:
        driver.quit()

def extract_gz(gz_path: str) -> str:
    """Decompress .gz file and save as .xml next to it. Returns xml path."""
    xml_path = gz_path.replace(".gz", "")
    print(f"Extracting {gz_path} → {xml_path}")
    with gzip.open(gz_path, "rb") as f_in:
        xml_text = f_in.read().decode("utf-8")
    with open(xml_path, "w", encoding="utf-8") as f_out:
        f_out.write(xml_text)
    os.remove(gz_path)  # cleanup the .gz after extraction
    print(f"Extracted → {xml_path}")
    return xml_path

def parse_locs(mode: int | None, xml_path: str) -> list[str]:
    """Parse all <loc> URLs from a sitemap XML file."""
    from bs4 import BeautifulSoup
    if mode == 1:
        with open(xml_path, "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f, "html.parser")
    elif mode == 2:
        with open(xml_path, "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f, "html.parser")
    urls = [tag.text.strip() for tag in soup.find_all("loc")]
    print(f"Found {len(urls)} URLs in {xml_path}")
    return urls

def trigger_download(driver, url: str, download_folder: str) -> str:
    """Navigate to a .gz URL, let Chrome auto-download it, return local path."""
    before = set(glob.glob(os.path.join(download_folder, "*.gz")))
    driver.get(url)
    # wait for NEW gz file to appear
    deadline = time.time() + 30
    while time.time() < deadline:
        after = set(glob.glob(os.path.join(download_folder, "*.gz")))
        tmp   = set(glob.glob(os.path.join(download_folder, "*.crdownload")))
        new   = after - before
        if new and not tmp:
            return max(new, key=os.path.getmtime)
        time.sleep(1)
    raise TimeoutError(f"Download timed out: {url}")

def parse_property_urls(xml_path: str, areas: list[str]) -> dict[str, list[str]]:
    """
    Parse a child sitemap XML file.
    - Only extract the id-id locale <loc> URL from each <url> block
    - Only keep URLs that contain /indonesia/
    - Filter by area slug
    """
    from bs4 import BeautifulSoup

    with open(xml_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    area_urls = {area: [] for area in areas}
    url_blocks = soup.find_all("url")
    print(f"  {len(url_blocks)} url blocks found in {os.path.basename(xml_path)}")

    for block in url_blocks:
        # Strategy: prefer xhtml:link with hreflang="id-id"
        # Fall back to main <loc> if not found
        id_id_tag = block.find("link", {"hreflang": "id-id"})

        if id_id_tag:
            url = id_id_tag.get("href", "").strip()
        else:
            loc = block.find("loc")
            url = loc.text.strip() if loc else ""

        if not url:
            continue

        # Only keep Indonesian properties
        if "/indonesia/" not in url.lower():
            continue

        # Filter by area slug
        url_lower = url.lower()
        for area in areas:
            area_slug = area.lower().replace(" ", "-")
            if area_slug in url_lower:
                area_urls[area].append(url)

    for area, urls in area_urls.items():
        if urls:
            print(f"  area '{area}': {len(urls)} properties")

    return area_urls

def scroll_to_explore(driver:webdriver.Chrome) -> None:
    # scroll to bottom to make sure we handle all lazy load elements
    last_height = driver.execute_script("return document.body.scrollHeight")
    current_position = 0

    while True:
        # Random chunk size (human scrolls vary between short and long flicks)
        chunk = random.randint(300, 700)
        current_position += chunk
        print(f"current_pos is : {current_position}")

        # Smooth scroll via JS scrollTo with 'smooth' behavior
        driver.execute_script(f"""
            window.scrollTo({{
                top: {current_position},
                behavior: 'smooth'
            }});
        """)

        # Random pause between scrolls (humans don't scroll at constant speed)
        time.sleep(random.uniform(0.4, 1.2))

        # Occasionally do a longer pause (like a human reading something)
        if random.random() < 0.2:   # 20% chance
            pause = random.uniform(1.5, 3.0)
            print(f"  [scroll] Natural reading pause: {pause:.1f}s")
            time.sleep(pause)

        # Occasionally scroll slightly back up (very human-like)
        if random.random() < 0.1:   # 10% chance
            scroll_back = random.randint(50, 150)
            print(f"  [scroll] Natural scroll back: {scroll_back:.1f}s")
            driver.execute_script(f"""
                window.scrollTo({{
                    top: {current_position - scroll_back},
                    behavior: 'smooth'
                }});
            """)
            time.sleep(random.uniform(0.3, 0.6))

        # Check if page height grew (new lazy content loaded)
        new_height = driver.execute_script("return document.body.scrollHeight")

        # If we've scrolled past current content bottom, check stability
        if current_position >= new_height:
            print(f"  [scroll] Scroll past current bottom")
            print(f"  [scroll] Current Position is : {current_position}")
            print(f"  [scroll] new_height 1 is : {new_height}")
            time.sleep(5)  # wait a beat for any final lazy loads
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                # One final slow scroll to absolute bottom
                print(f"  [scroll] Scroll make sure 3 times")
                for _ in range(3):
                    driver.execute_script("""
                        window.scrollTo({
                            top: document.body.scrollHeight,
                            behavior: 'smooth'
                        });
                    """)
                    new_height = driver.execute_script("return document.body.scrollHeight")
                    time.sleep(5)
                last_height = new_height
                if new_height == last_height:
                    print("  [scroll] Reached stable bottom.")
                    break
            # Page grew — update and keep scrolling
            last_height = new_height

def sub_explorer(driver: webdriver.Chrome) -> list[str]:
    hrefs = []
    if os.path.exists("seen_url.json"):
        print("JSON Found")
        with open("seen_url.json", "r") as f:
            seen_url_ref = json.load(f)
    else:
        seen_url_ref = {}
        with open("seen_url.json", "w") as f:
            json.dump(seen_url_ref, f)
        log.info("Created fresh seen_url.json")

    cards = WebDriverWait(driver, PAGE_TIMEOUT).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, "a[data-testid='seo-card']"))
    )
    for card in cards:
        href = card.get_attribute("href")
        href_fix = re.sub(r'-\d+$', '', href)
        if href_fix not in seen_url_ref.keys():
            seen_url_ref[href_fix] = ""
            hrefs.append(href_fix)
        else:
            continue

    with open("seen_url.json", "w") as f:
        json.dump(seen_url_ref, f)
    
    # href_master_list.extend(hrefs)
    return hrefs

def scrape_sitemaps(url: str, driver) -> list[str]:
    if url == "" or url == None:
        return []

    try:
        # print("Visiting homepage to warm up session...")
        # driver.get("https://www.tiket.com")
        # time.sleep(10)

        # if not headless:
        #     input(">>> Solve Cloudflare challenge if it appears, then press ENTER ...")

        # Trigger the download
        driver.get(url)
        time.sleep(20)
        # Wait up to 10 seconds for the element to appear
        # property_card = WebDriverWait(driver, PAGE_TIMEOUT).until(
        #     EC.presence_of_element_located((By.CSS_SELECTOR, "div[data-testid='seo-additional-content']"))
        # )
        property_card = WebDriverWait(driver, PAGE_TIMEOUT).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "a[data-testid='seo-card']"))
        )
        print("property card found!!!")
    except Exception as e:
        print("Element not found within timeout:", e)
    
    href_master_list = []
    if property_card:
        scroll_to_explore(driver)
        time.sleep(5)

        style1 = driver.find_element(
            By.CSS_SELECTOR, "div.main_pagination_container__gX6WA"
        )

        # STYLE2 COULD HAVE MULTIPLE sections
        style2 = driver.find_elements(
            By.CSS_SELECTOR, "div.SeoAdditionalContent_dots__MLHpe"
        )

        if style1:
            print("Style 1 found, locating next page buttons")
            potential_page = driver.find_elements(
                By.CSS_SELECTOR, "a.Pagination_button__xnJqp.Pagination_anchor__rX_07"
            )
            page_numbers = []
            for el in potential_page:
                text = el.text.strip()
                if text.isdigit():  # check if numeric
                    page_numbers.append(int(text))
            max_page = max(page_numbers) if page_numbers else None
            print(f"Found {max_page} potential pages.")

            for i in range(0, max_page):
                if i == max_page: 
                    print("We're reaching final page, breaking.")
                    break

                print(f"Exploring page {i+1} out of {max_page} pages")
                buffer_list = sub_explorer(driver)
                href_master_list.extend(buffer_list)
                print(f"[A] Successfully scrapped {len(buffer_list)} URL")
                print(f"[A] total collected URL is {len(href_master_list)} URL")

                button_inspectors = driver.find_elements(
                    By.CSS_SELECTOR, "a.Pagination_button__xnJqp.Pagination_anchor__rX_07"
                )
                # access the next page which usually located on the last sequence
                href_value = button_inspectors[-1].get_attribute("href")
                print("Moving to next page.")
                driver.get(href_value)
                time.sleep(5)

        if len(style2) > 0:
            print("Style 2 found, locating next page buttons")
            print(f"We have {len(style2)} sections.")
            wait = WebDriverWait(driver, PAGE_TIMEOUT)

            # for i in range(len(style2)):
            #     print(f"Processing section no {i}")
            time.sleep(5)
            buttons_style2 = wait.until(
                EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "button[aria-label^='page-control-dot']")
                )
            )

            total = len(buttons_style2)
            for i in range(total):
                print(f"processing loop {i+1}")
                try:
                    # Re-fetch buttons every loop (IMPORTANT: avoid stale elements)
                    buttons = driver.find_elements(By.CSS_SELECTOR, "button[aria-label^='page-control-dot']")
                    
                    btn = buttons[i]

                    # Scroll into view (optional but useful)
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
                    time.sleep(5)
                    # Click using JS (more reliable for UI-heavy sites)
                    driver.execute_script("arguments[0].click();", btn)
                    time.sleep(3)
                    
                    # Wait for carousel state change
                    # Option 1: wait until this button becomes active
                    wait.until(lambda d: "SeoAdditionalContent_active__gpRv3" in btn.get_attribute("class"))

                    # Option 2 (better): wait for content to change (if identifiable)
                    # e.g. wait until image/src/text changes

                    time.sleep(1)  # small buffer (can replace with smarter wait)
                    buffer_list = sub_explorer(driver)
                    href_master_list.extend(buffer_list)
                    print(f"[B] Scraped {len(buffer_list)} URLs at Page {i+1}")
                    print(f"[B] total collected URL is {len(href_master_list)} URL")
                except Exception as e:
                    print(f"  error: {e}")
                    continue
    return href_master_list

def extract_indonesia_urls(xml_file):
    bali_area = ["bali", "ubud", "seminyak", "canggu", "kintamani", "denpasar", "tabanan", "sidemen",
                 "karangasem", "lovina", "bedugul", "nusa-dua", "nusa dua", "nusadua", "sanur", "kuta", 
                 "menjangan", "buyan", "batur"]
    tree = ET.parse(xml_file)
    root = tree.getroot()

    namespace = {"ns": "http://www.sitemaps.org/schemas/sitemap/0.9"}

    target_prefix = "https://www.tiket.com/id-id/homes/indonesia"

    results = []
    if os.path.exists("seen_url.json"):
        print("JSON Found")
        with open("seen_url.json", "r") as f:
            seen_url_ref = json.load(f)
    else:
        seen_url_ref = {}
        with open("seen_url.json", "w") as f:
            json.dump(seen_url_ref, f)
        log.info("Created fresh seen_url.json")

    for url in root.findall("ns:url", namespace):
        loc = url.find("ns:loc", namespace)
        if loc is not None:
            link = loc.text.strip()
            link_fix = re.sub(r'-\d+$', '', link)
            filter_url = any(keyword in link_fix for keyword in bali_area)
            if filter_url and (link_fix.startswith(target_prefix)) and (
                link_fix not in seen_url_ref.keys()):
                seen_url_ref[link_fix] = ""
                results.append(link_fix)

    with open("seen_url.json", "w") as f:
        json.dump(seen_url_ref, f)
    return results

def get_checkin_checkout(offset_days: int = 1) -> tuple[str, str]:
    """
    Return (checkin, checkout) as ISO strings.
    Default: tomorrow → day after tomorrow (1-night stay).
    """
    checkin  = datetime.today() + timedelta(days=offset_days)
    checkout = checkin + timedelta(days=1)
    return checkin.strftime("%Y-%m-%d"), checkout.strftime("%Y-%m-%d")

def renew_checkin_checkout(cur_date, increment_day: int = 1) -> tuple[str, str]:
    """
    Return new (checkin, checkout) as ISO strings.
    Renew dates due to availability problem.
    """
    cur_date_clean = datetime.strptime(cur_date, "%Y-%m-%d")
    checkin  = cur_date_clean + timedelta(days=increment_day)
    checkout = checkin + timedelta(days=1)
    return checkin.strftime("%Y-%m-%d"), checkout.strftime("%Y-%m-%d")

def build_search_url(incoming_url, checkin: str, checkout: str) -> str:
    """Build minimal, tracking-free tiket.com search URL."""
    base = incoming_url
    params = (
        # f"&dest_type={DEST_TYPE}"
        f"?checkin={checkin}"
        f"&checkout={checkout}"
        f"&adult=2"
        f"&room=1"
        f"&night=1"
    )
    # f"&dest_id={DEST_ID}"
    return base + params

def scrape_hotel_price(
    driver: webdriver.Chrome,
    hotel_url: str,
) -> dict | None:
    """
    Visit a hotel page with date params and extract the displayed price.
    Returns a record dict or None if extraction fails.
    """
    # SUPPORTING FUNCTIONS ======================================
    def safe_text(parent, by, selector):
        """
        Safely extract text from element.
        Return None if element is not found.
        """
        try:
            return parent.find_element(by, selector).text.strip()
        except NoSuchElementException:
            return None

    def safe_texts(parent, by, selector):
        """
        Safely extract multiple texts from elements.
        Return None if elements are not found.
        """
        try:
            elements = parent.find_elements(by, selector)
            texts = [el.text.strip() for el in elements if el.text.strip()]
            return texts if texts else None
        except:
            return None
    # END OF SUPPORTING FUNCTIONS =====================================

    ckin_dt, ckout_dt = get_checkin_checkout()
    hotel_url_new = build_search_url(hotel_url, ckin_dt, ckout_dt)
    dt = datetime.strptime(ckin_dt, "%Y-%m-%d")  # use your actual format
    day_weekday = dt.strftime("%A")

    dt_now = datetime.today()
    dt_now_rev = dt_now.strftime("%Y-%m-%d")

    print(f"  Visiting: {hotel_url_new}")
    property_name = None

    date_avail = False
    only_once_pname = True
    new_ckin = ckin_dt
    trials_counter = 0
    while date_avail == False:
        driver.get(hotel_url_new)
        wait = WebDriverWait(driver, PAGE_TIMEOUT)

        if only_once_pname == True: # FIND PROPERTY NAME
            try:
                hotel_name_elem = wait.until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "div.HotelInfo_property_name__qba65 h1[data-testid='name']")
                    )
                )
                property_name = hotel_name_elem.text.strip() or None
                print(f"property name is: {property_name}")
            except (TimeoutException, NoSuchElementException):
                property_name = None
            only_once_pname = False

        # CHECK IF DATE AVAILABLE OR NOT
        try:
            try:
                container = wait.until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "div.RoomListErrorView_wrapper__jy94T")
                    )
                )
                if container:
                    trials_counter += 1
                    if trials_counter > 5:
                        time.sleep(5)
                        raise RuntimeError("Trials changing date exceeding limits (5) times.")

                    print("The availability on current date is not available, renew the date range.")
                    new_ckin, new_ckout = renew_checkin_checkout(new_ckin)
                    hotel_url_new = build_search_url(hotel_url, checkin=new_ckin, checkout=new_ckout)
                    time.sleep(7)
                    continue
            except (TimeoutException, NoSuchElementException):
                print("Room may be available!!!!")
        except (TimeoutException, NoSuchElementException):
            print("Unknown exceptions, can't find any booking data at all, check actual conditions.")
            results = []
            data = {"room_type": None, "price": None, "notes": None, "weekday": day_weekday, "date_scrapped": dt_now_rev}
            results.append(data)
            return {"property_name": property_name, "total_rows": 0, "rows": results}
        except RuntimeError as e:
            print(f"Caught a runtime error: {e}")
            results = []
            data = {"room_type": None, "price": None, "notes": None, "weekday": day_weekday, "date_scrapped": dt_now_rev}
            results.append(data)
            return {"property_name": property_name, "total_rows": 0, "rows": results}

        # find section with room informations
        try:
            driver.find_element(
                By.CSS_SELECTOR,
                "div.MainRoomGroupLists_room_group_lists_container__04n19"
            )
        except NoSuchElementException:
            # sometimes table rows are direct children; fallback to container
            print("NoSuchElementException (main) triggered!!! Fallback to sending just NaN")
            return {"property_name": property_name, "total_rows": 0, "rows": results}
            
        time.sleep(5)
        rows = driver.find_elements(By.CSS_SELECTOR, "section[data-testid='room-group-item-section']")
        print(f"Total room rows found: {len(rows)}")

        # Wait until ALL room title containers appear
        # title_containers = WebDriverWait(driver, 10).until(
        #     EC.presence_of_element_located((
        #         By.CSS_SELECTOR,
        #         "div.FacilitiesNudges_mobile_room_group_expanded_title__EmlX5"
        #     ))
        # )

        results = []
        for _, row in enumerate(rows, start=1):
            data = {"room_type": None, "price": None, "notes": None, "weekday": day_weekday, "date_scrapped": dt_now_rev}
            try:
                room_cards = row.find_element(
                    By.CSS_SELECTOR,
                    "div.RoomGroupItem_room_group_lists_main_container__CZ7DL"
                )
                time.sleep(5)
                # ROOM TYPE -----------------------------------------------------------------------
                try:
                    room_type = row.text.split("\n")[0]
                except Exception as e:
                    print(f"Error due to {e}")
                
                print(f"room_type name is: {room_type}")
                time.sleep(5)
                # FEATURES -----------------------------------------------------------------------
                features = []

                # Bed type
                bed_type = safe_text(
                    room_cards,
                    By.CSS_SELECTOR,
                    "div.BedTypesContent_bed_types_main_content_wrapper__c8T0W span"
                )
                if bed_type:
                    features.append(bed_type)
                print(f"bed_type name is: {bed_type}")

                # Room nudges/features
                extra_features = safe_texts(
                    room_cards,
                    By.CSS_SELECTOR,
                    "div.FacilitiesNudges_room_group_variables_info_item__io0wb span"
                )
                if extra_features:
                    features.extend(extra_features)
                print(f"extra_features name is: {extra_features}")

                # Breakfast labels
                breakfast = safe_texts(
                    room_cards,
                    By.CSS_SELECTOR,
                    "div.FeatureLabels_feature_labels_container__6SJTk span"
                )
                if breakfast:
                    features.extend(breakfast)
                print(f"breakfast name is: {breakfast}")

                # Cancellation policy
                cancellation = safe_text(
                    room_cards,
                    By.CSS_SELECTOR,
                    "span[data-testid='cancellation-policies-title']"
                )
                if cancellation:
                    features.append(cancellation)

                # Value-added section
                bonuses = safe_texts(
                    room_cards,
                    By.CSS_SELECTOR,
                    "div.ValueAddedSection_value_added_section_container__812zf span"
                )
                if bonuses:
                    features.extend(bonuses)

                # Remove duplicates
                if features:
                    features = list(dict.fromkeys(features))
                else:
                    features = None
                time.sleep(5)

                # DISCOUNTED PRICE -----------------------------------------------------------------------
                discounted_price = safe_text(
                    room_cards,
                    By.CSS_SELECTOR,
                    "div.RatePlanPrice_final_price_wrapper__t1r9i div.Text_variant_price__wu_WD"
                )
                time.sleep(5)
                print(f"discounted_price is: {discounted_price}")

                # STORE RESULT ---------------------------------------------------------------------------
                data["room_type"] = room_type
                data["price"] = discounted_price
                data["notes"] = features
                results.append(data)
            except Exception as e:
                print(f" [A] error: {e}")
                continue
        date_avail = True
    
    return {"property_name": property_name, "total_rows": len(rows), "rows": results}


# ── main pipeline ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # areas = ["Ubud", "Canggu", "Tabanan", "Seminyak", "Kuta", "Denpasar", "Kintamani",
    #         "Bedugul", "Karangasem", "Nusa Dua", "Sanur"]
    areas = ["Ubud", "Canggu", "Tabanan", "Seminyak", "Kuta", "Kintamani",
            "Bedugul", "Karangasem"]

    PROJECT_FOLDER   = os.path.join(os.getcwd(), "sitemaps")
    CHILD_FOLDER     = os.path.join(os.getcwd(), "child-sitemaps")
    OTHER_CHILD_FOLDER = os.path.join(os.getcwd(), "other-child-sitemaps")
    CURR_DIR         = os.path.dirname(os.getcwd())
    SITEMAP_INDEX    = "https://www.tiket.com/sitemap/id-id/index.xml.gz"

    # Auto-detect VPS (Linux without display) → force headless
    IS_VPS = platform.system() == "Linux" and not os.environ.get("DISPLAY")
    print(f"Running in {'headless (VPS)' if IS_VPS else 'headed (local)'} mode")
    aggregate_url = False #ACTIVATE IF YOU WANT TO UPDATE URL LISTS

    if aggregate_url == True:
        # Step 1 — download index.xml.gz
        gz_path = download_sitemap(SITEMAP_INDEX, PROJECT_FOLDER, headless=IS_VPS)

        # Step 2 — extract gz → xml
        xml_path = extract_gz(gz_path)

        # Step 3 — parse child sitemap URLs
        child_urls = parse_locs(1, xml_path)

        # Step 4 — filter & print (replace with your next pipeline step)
        PDP_KEYWORDS = ["hotel-pdp", "homes-pdp", "homes-villa", "homes-glamping", "homes-cottage"]
        relevant = [u for u in child_urls if any(k in u for k in PDP_KEYWORDS)]
        print(f"\nRelevant child sitemaps: {len(relevant)}")
        for u in relevant:
            print(f"  {u}")

        # There will be 2 possibilities: sitemaps that have "...area.xml" or "...city.xml" or "...region.xml"
        # will be scrapped traditionally by using scrolling and click
        # However, sites that doesn't contain those elements already have ready to access url
        
        print("\n[Step 3] Downloading child sitemaps...")
        debug_counter = 0
        second_filter = ["area", "city", "region"]
        aggregate_url = []
        for i, child_url in enumerate(relevant, 1):
            debug_counter += 1
            property_urls = {area: [] for area in areas}
            if debug_counter > 5: break
            print(f"  [{i}/{len(relevant)}] {child_url.split('/')[-1]}")
            try:
                filter_url = any(keyword in child_url for keyword in second_filter)
                if filter_url:
                    # print(f"[DEBUG] Skipping filtered sections and move on to villas-pdp section.")
                    # continue
                    gz_child = download_sitemap(child_url, CHILD_FOLDER, headless=IS_VPS)
                    xml_child = extract_gz(gz_child)
                    locs_child = parse_locs(2, xml_child)

                    found = parse_property_urls(xml_child, areas)
                    for area in areas:
                        property_urls[area].extend(found[area])

                    # step 5 - ikutin pipeline booking.com -> eksplorasi 
                    driver_next = build_driver(CURR_DIR, headless=IS_VPS)
                    for k, v in property_urls.items():
                        print(f"Focusing search on area {k}")
                        print(f"With URL {v}")
                        if len(v) > 0:
                            list_properties = scrape_sitemaps(v[0], driver_next)
                        else:
                            list_properties = []
                        aggregate_url.extend(list_properties)
                        print(f"Total URL captured is {len(aggregate_url)} URL")

                    if os.path.exists("URL_pool_list.txt"):
                        for i in aggregate_url:
                            with open("URL_pool_list.txt", "a") as file:
                                file.write(f"{i}\n")
                    else:
                        raise RuntimeError("No URL_pool_list.txt found, make it first please.")
                    driver_next.quit()
                else:
                    gz_child = download_sitemap(child_url, OTHER_CHILD_FOLDER, headless=IS_VPS)
                    xml_child = extract_gz(gz_child)
                    print(f"[DEBUG] xml child is : {xml_child}")
                    locs_child = extract_indonesia_urls(xml_child)
                    print(f"[DEBUG] locs child is : {locs_child[0]}")
                    print(f"[DEBUG] locs child length is : {len(locs_child)}")
                    # WE ALREADY GOT THE URL SO JUST DIRECTLY PUT IT TO TXT FILE!!!
                    if os.path.exists("URL_pool_list.txt"):
                        for i in locs_child:
                            with open("URL_pool_list.txt", "a") as file:
                                file.write(f"{i}\n")
            except Exception as e:
                print(f" [A] error: {e}")
                continue

    # ----- OPEN THE URL_POOL_LIST.TXT AND START OPENING ONE BY ONE
    try:
        driver = build_driver(CURR_DIR, headless=IS_VPS)
        new_urls = []
        debug_counter = 0
        allowed_proceed = False
        only_once = True
        break_signal = False
        if os.path.exists("checkpoint_save.txt"):
            with open("checkpoint_save.txt", "r") as f:
                cross_check = f.read()
                print(f"cross_check is : {cross_check}")

        with open("URL_pool_list.txt", "r") as file:
            lines = file.readlines()
            last_line = lines[-1].strip()
            # print(f"[DEBUG] last_line is {last_line}")

            i = 0
            while i < len(lines):
                # print(f"[DEBUG] lines[i] is {lines[i]}")
                if lines[i] == last_line:
                    i = 0
                    break_signal = True
                
                if only_once == True and os.path.exists("checkpoint_save.txt"):
                    result = bool(re.search(str(lines[i]), str(cross_check)))
                    # print(f"[DEBUG] result is {result}")
                    if result == True:
                        print("Found last checkpoint!!!")
                        allowed_proceed = True
                        only_once = False
                        continue

                if allowed_proceed == True or not os.path.exists("checkpoint_save.txt"):
                    new_urls.append(lines[i])
                    debug_counter += 1

                i += 1
                if break_signal == True or debug_counter > MAX_PROPERTIES: 
                    print("Safe guarding with 15 properties limiter, breaking algorithm now!")
                    break

        targets = new_urls[:MAX_PROPERTIES]
        print(f"Targeting {len(targets)} hotels this session.")

        new_records = []
        for i, url in enumerate(targets, 1):
            print(f"[{i}/{len(targets)}] Scraping …")
            record = scrape_hotel_price(driver, url)
            time.sleep(3)
            if record:
                new_records.append(record)
                with open("checkpoint_save.txt", "w") as f:
                    f.write(f"{url}\n")

        print(f"[DEBUG] check scraping result: \n{new_records}")
    except Exception as e:
        print(f" [A] error: {e}")
    finally:
        # driver.quit()
        log.info("Driver shut down. Session END.")
        log.info("=" * 50)   


Running in headed (local) mode
Safe guarding with 15 properties limiter, breaking algorithm now!
Targeting 15 hotels this session.
[1/15] Scraping …
  Visiting: https://www.tiket.com/id-id/homes/indonesia/1br-private-pool-villa-in-kerobokan-north-kuta
?checkin=2026-05-09&checkout=2026-05-10&adult=2&room=1&night=1
property name is: 1BR Private Pool Villa in Kerobokan North Kuta
Room may be available!!!!
Total room rows found: 1
room_type name is: Vila 1 Kamar Tidur
bed_type name is: Single
extra_features name is: None
breakfast name is: ['Sarapan tidak tersedia']
discounted_price is: IDR 1.299.019
[2/15] Scraping …
  Visiting: https://www.tiket.com/id-id/homes/indonesia/2br-private-studios-package-in-seminyak
?checkin=2026-05-09&checkout=2026-05-10&adult=2&room=1&night=1
property name is: 2BR Private Studios package in Seminyak
The availability on current date is not available, renew the date range.
The availability on current date is not available, renew the date range.
The availabilit

17:34:59 [INFO] Driver shut down. Session END.
17:34:59 [INFO] ==================================================


[DEBUG] check scraping result: 
[{'property_name': '1BR Private Pool Villa in Kerobokan North Kuta', 'total_rows': 1, 'rows': [{'room_type': 'Vila 1 Kamar Tidur', 'price': 'IDR 1.299.019', 'notes': ['Single', 'Sarapan tidak tersedia', 'Tidak bisa refund & reschedule', 'Kopi & teh, Parkir'], 'weekday': 'Saturday', 'date_scrapped': '2026-05-08'}]}, {'property_name': '2BR Private Studios package in Seminyak', 'total_rows': 0, 'rows': [{'room_type': None, 'price': None, 'notes': None, 'weekday': 'Saturday', 'date_scrapped': '2026-05-08'}]}, {'property_name': '2BR Dharman Villa Canggu', 'total_rows': 0, 'rows': [{'room_type': None, 'price': None, 'notes': None, 'weekday': 'Saturday', 'date_scrapped': '2026-05-08'}]}, {'property_name': '1BED ROOM PRIVATE POOL VILLA @ SAWAH UBUD', 'total_rows': 0, 'rows': [{'room_type': None, 'price': None, 'notes': None, 'weekday': 'Saturday', 'date_scrapped': '2026-05-08'}]}, {'property_name': '2 Zen Bali Villa Canggu', 'total_rows': 0, 'rows': [{'room_type

In [48]:
data_df = pd.DataFrame()
for i in new_records:
    room_type = []
    price_details = []
    notes_details = []
    property_list = []
    dt_scrapped_list = []
    weekday_list = []
    sub_data = {}
    for ii in range(len(i['rows'])):
        property_list.append(i['property_name'])
        dt_scrapped_list.append(i['rows'][ii]['date_scrapped'])
        if i['rows'][ii]['room_type'] is not None:
            room_type.append(i['rows'][ii]['room_type'])
        else:
            room_type.append(None)

        if i['rows'][ii]['price'] is not None:
            price_details.append(i['rows'][ii]['price'])
        else:
            price_details.append(None)

        if i['rows'][ii]['notes'] is not None:
            notes_details.append(i['rows'][ii]['notes'])
        else:
            notes_details.append(None)

        if i['rows'][ii]['weekday'] is not None:
            weekday_list.append(i['rows'][ii]['weekday'])
        else:
            weekday_list.append(None)
    sub_data['property_name'] = property_list
    sub_data['room_type'] = room_type
    sub_data['price_details'] = price_details
    sub_data['notes_details'] = notes_details
    sub_data['weekday'] = weekday_list
    sub_data['date_scrapped'] = dt_scrapped_list
    major_data = pd.DataFrame(sub_data)
    data_df = pd.concat([data_df, major_data])

data_df

,property_name,room_type,price_details,notes_details,weekday,date_scrapped
0,1BR Private Pool Villa in Kerobokan North Kuta,Vila 1 Kamar Tidur,IDR 1.299.019,"[Single, Sarapan tidak tersedia, Tidak bisa re...",Saturday,2026-05-08
0,2BR Private Studios package in Seminyak,None,None,None,Saturday,2026-05-08
0,2BR Dharman Villa Canggu,None,None,None,Saturday,2026-05-08
0,1BED ROOM PRIVATE POOL VILLA @ SAWAH UBUD,None,None,None,Saturday,2026-05-08
0,2 Zen Bali Villa Canggu,None,None,None,Saturday,2026-05-08
0,1 Bed Room Bali House in Nusa Dua,None,None,None,Saturday,2026-05-08
0,2Bedroom - Kimmemore Villa by SooBali - Ubud,Vila 2 Kamar Tidur,IDR 1.390.905,"[Double, Double, Kamar bebas asap rokok, Sarap...",Saturday,2026-05-08
0,2BR (No Kitchen) Apartment at Skales Residence...,None,None,None,Saturday,2026-05-08
0,2BR Villa Alba in Canggu by OriVista,Kamar Deluks Vila 1 Kamar Tidur dengan Pemanda...,IDR 1.698.346,"[Queen, Boleh merokok di kamar, Sarapan tidak ...",Saturday,2026-05-08
0,1 BEDROOM VILLA AT KUTA SEMINYAK,None,None,None,Saturday,2026-05-08


In [50]:
import numpy as np

data_df2 = data_df.copy()
data_df2 = data_df2.reset_index(drop=True)

data_df2["price_details"] = (
    data_df2["price_details"]
    .astype(str)
    .str.replace("IDR", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.strip()
)

data_df2["price_details"] = pd.to_numeric(data_df2["price_details"], errors="coerce")
data_df2["price_details"] = data_df2["price_details"].fillna(0).astype(int)
data_df2

,property_name,room_type,price_details,notes_details,weekday,date_scrapped
0,1BR Private Pool Villa in Kerobokan North Kuta,Vila 1 Kamar Tidur,1299019,"[Single, Sarapan tidak tersedia, Tidak bisa re...",Saturday,2026-05-08
1,2BR Private Studios package in Seminyak,None,0,None,Saturday,2026-05-08
2,2BR Dharman Villa Canggu,None,0,None,Saturday,2026-05-08
3,1BED ROOM PRIVATE POOL VILLA @ SAWAH UBUD,None,0,None,Saturday,2026-05-08
4,2 Zen Bali Villa Canggu,None,0,None,Saturday,2026-05-08
5,1 Bed Room Bali House in Nusa Dua,None,0,None,Saturday,2026-05-08
6,2Bedroom - Kimmemore Villa by SooBali - Ubud,Vila 2 Kamar Tidur,1390905,"[Double, Double, Kamar bebas asap rokok, Sarap...",Saturday,2026-05-08
7,2BR (No Kitchen) Apartment at Skales Residence...,None,0,None,Saturday,2026-05-08
8,2BR Villa Alba in Canggu by OriVista,Kamar Deluks Vila 1 Kamar Tidur dengan Pemanda...,1698346,"[Queen, Boleh merokok di kamar, Sarapan tidak ...",Saturday,2026-05-08
9,1 BEDROOM VILLA AT KUTA SEMINYAK,None,0,None,Saturday,2026-05-08


In [51]:
import pickle

if os.path.exists("scraping_database_tiket.pkl"):
    loaded_df = pd.read_pickle("scraping_database_tiket.pkl")
    data_df2 = pd.concat([data_df2, loaded_df])
    data_df2.to_pickle("scraping_database_tiket.pkl")

else:
    data_df2.to_pickle("scraping_database_tiket.pkl")

data_df2.reset_index(drop=True)

,property_name,room_type,price_details,notes_details,weekday,date_scrapped
0,1BR Private Pool Villa in Kerobokan North Kuta,Vila 1 Kamar Tidur,1299019,"[Single, Sarapan tidak tersedia, Tidak bisa re...",Saturday,2026-05-08
1,2BR Private Studios package in Seminyak,None,0,None,Saturday,2026-05-08
2,2BR Dharman Villa Canggu,None,0,None,Saturday,2026-05-08
3,1BED ROOM PRIVATE POOL VILLA @ SAWAH UBUD,None,0,None,Saturday,2026-05-08
4,2 Zen Bali Villa Canggu,None,0,None,Saturday,2026-05-08
5,1 Bed Room Bali House in Nusa Dua,None,0,None,Saturday,2026-05-08
6,2Bedroom - Kimmemore Villa by SooBali - Ubud,Vila 2 Kamar Tidur,1390905,"[Double, Double, Kamar bebas asap rokok, Sarap...",Saturday,2026-05-08
7,2BR (No Kitchen) Apartment at Skales Residence...,None,0,None,Saturday,2026-05-08
8,2BR Villa Alba in Canggu by OriVista,Kamar Deluks Vila 1 Kamar Tidur dengan Pemanda...,1698346,"[Queen, Boleh merokok di kamar, Sarapan tidak ...",Saturday,2026-05-08
9,1 BEDROOM VILLA AT KUTA SEMINYAK,None,0,None,Saturday,2026-05-08


In [4]:
aggregate_url

['https://www.tiket.com/id-id/homes/indonesia/woywoy-escape-408001630224923632',
 'https://www.tiket.com/id-id/homes/indonesia/kanva-ubud-509001664166638091',
 'https://www.tiket.com/id-id/homes/indonesia/ubud-mas-glamping-luxury-tent-707001721964027238',
 'https://www.tiket.com/id-id/homes/indonesia/firefly-eco-lodge-903001772684662055',
 'https://www.tiket.com/id-id/homes/indonesia/woywoy-escape-408001630224923632',
 'https://www.tiket.com/id-id/homes/indonesia/ubud-mas-glamping-luxury-tent-707001721964027238',
 'https://www.tiket.com/id-id/homes/indonesia/sandat-glamping-tents-108001534518416814',
 'https://www.tiket.com/id-id/homes/indonesia/kanva-ubud-509001664166638091',
 'https://www.tiket.com/id-id/homes/indonesia/ubud-mas-glamping-luxury-tent-807001751994620595',
 'https://www.tiket.com/id-id/homes/indonesia/firefly-eco-lodge-903001772684662055',
 'https://www.tiket.com/id-id/homes/indonesia/ubud-tropical-glamping-903001773725692794',
 'https://www.tiket.com/id-id/homes/indone

In [ ]:
# reference link:
# https://www.tiket.com/id-id/homes/indonesia/woywoy-escape-408001630224923632
# https://www.tiket.com/id-id/homes/indonesia/kanva-ubud-509001664166638091
# https://www.tiket.com/id-id/homes/indonesia/woywoy-escape?checkin=2026-05-09&checkout=2026-05-10&adult=1&room=1&night=1

In [4]:
import pickle
import pandas as pd

df = pd.read_pickle("scraping_database_tiket_eb.pkl")
df[df['property_name'] == "Alam Terrace Cottages"]

,property_name,room_type,ota_source,price_details,notes_details,target_date,category,date_scrapped,villa_or_hotel,rating
0,Alam Terrace Cottages,Double Superior Room with Garden View,tiket.com,513491,"[Double, Smoking room, No breakfast, 100% Refu...",2026-07-29,early-book,2026-07-13,1,3
1,Alam Terrace Cottages,Double Superior Room with Garden View,tiket.com,591645,"[Double, Smoking room, No breakfast, 100% Refu...",2026-08-12,early-book,2026-07-13,1,3
2,Alam Terrace Cottages,Double Superior Room with Garden View,tiket.com,513491,"[Double, Smoking room, No breakfast, 100% Refu...",2026-07-29,early-book,2026-07-13,1,3
3,Alam Terrace Cottages,Double Superior Room with Garden View,tiket.com,591645,"[Double, Smoking room, No breakfast, 100% Refu...",2026-08-12,early-book,2026-07-13,1,3
4,Alam Terrace Cottages,Double Superior Room with Garden View,tiket.com,513491,"[Double, Smoking room, No breakfast, 100% Refu...",2026-07-29,early-book,2026-07-13,1,3
5,Alam Terrace Cottages,Double Standard Room with Garden View,tiket.com,506344,"[Double, Smoking room, No breakfast, 100% Refu...",2026-08-09,early-book,2026-07-13,1,3
6,Alam Terrace Cottages,None,tiket.com,0,None,2026-07-29,early-book,2026-07-13,1,3
7,Alam Terrace Cottages,Double Standard Room with Garden View,tiket.com,506344,"[Double, Smoking room, No breakfast, 100% Refu...",2026-08-09,early-book,2026-07-13,1,3
162,Alam Terrace Cottages,None,tiket.com,0,None,2026-07-29,early-book,2026-07-13,1,3
163,Alam Terrace Cottages,Double Superior Room with Garden View,tiket.com,0,"[Double, Smoking room, No breakfast, 100% Refu...",2026-08-12,early-book,2026-07-13,1,3


In [1]:
import argparse
import csv
import json
import logging
import random, re, shutil, tempfile, psutil, subprocess
import gzip, os, time, glob, platform
from dataclasses import asdict, dataclass, field
from datetime import datetime, timedelta
from io import BytesIO
from pathlib import Path
from typing import Optional
from urllib.parse import urlparse
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse
import pandas as pd
from difflib import SequenceMatcher
 
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager
import xml.etree.ElementTree as ET
import undetected_chromedriver as uc
from match_url_tiket import main_processing

# ── logging ────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("tiket_scraper.log", encoding="utf-8"),
    ],
)
log = logging.getLogger(__name__)

# ── constants ──────────────────────────────────────────────────────────────────
SITEMAP_INDEX_URL = "https://www.tiket.com/sitemap/id-id/index.xml.gz"
BASE_URL          = "https://www.tiket.com"
LOCALE            = "id-id"
PAGE_TIMEOUT      = 20
MAX_PROPERTIES    = 25
DRIVER_RECYCLE_EVERY = 5
# Linux → VPS mode (Xvfb); anything else (e.g. local Windows) → headed local mode
IS_VPS = platform.system() == "Linux"
 
# Child sitemap name patterns that contain property detail pages (PDP)
PDP_PATTERNS = {
    "hotel":     ["hotel-pdp"],
    "villa":     ["homes-villa"],
    "homes":     ["homes-pdp"],
    "glamping":  ["homes-glamping"],
    "cottage":   ["homes-cottage"],
    "apartment": ["homes-apartment"],
}

def ensure_virtual_display():
    """
    Start Xvfb virtual display if not already running.
    Sets DISPLAY env var so Chrome picks it up automatically.
    """
    display = ":99"

    # Check if Xvfb is already running on this display
    result = subprocess.run(
        ["pgrep", "-f", f"Xvfb {display}"],
        capture_output=True, text=True
    )

    if result.returncode != 0:
        # Not running — start it
        log.info(f"Starting Xvfb on display {display}...")
        subprocess.Popen(
            ["Xvfb", display, "-screen", "0", "1920x1080x24"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
         # ── Wait until Xvfb socket actually exists, not just a fixed sleep ──
        socket_path = f"/tmp/.X11-unix/X{display.lstrip(':')}"
        for _ in range(20):          # up to 10s
            if os.path.exists(socket_path):
                log.info(f"Xvfb socket ready: {socket_path}")
                break
            time.sleep(0.5)
        else:
            raise RuntimeError("Xvfb did not start in time — socket never appeared")
    else:
        log.info(f"Xvfb already running on {display}")

    # CRITICAL: set DISPLAY so Chrome finds the virtual screen
    os.environ["DISPLAY"] = display
    log.info(f"DISPLAY set to {display}")

def get_chrome_version() -> int:
    """Auto-detect installed Chrome major version — no more hardcoding version_main."""
    # ── Windows: registry first, then chrome.exe file metadata ────────────────
    if platform.system() == "Windows":
        try:
            import winreg
            with winreg.OpenKey(winreg.HKEY_CURRENT_USER, r"Software\Google\Chrome\BLBeacon") as key:
                version_str, _ = winreg.QueryValueEx(key, "version")  # e.g. "149.0.7827.201"
                major = int(version_str.split(".")[0])
                log.info(f"Auto-detected Chrome (registry): {version_str} → major version {major}")
                return major
        except Exception as e:
            log.warning(f"Registry Chrome detection failed: {e}")

        # Fallback: read version straight from chrome.exe file metadata
        chrome_paths = [
            os.path.expandvars(r"%ProgramFiles%\Google\Chrome\Application\chrome.exe"),
            os.path.expandvars(r"%ProgramFiles(x86)%\Google\Chrome\Application\chrome.exe"),
            os.path.expandvars(r"%LocalAppData%\Google\Chrome\Application\chrome.exe"),
        ]
        for path in chrome_paths:
            if os.path.exists(path):
                try:
                    result = subprocess.run(
                        ["powershell", "-NoProfile", "-Command",
                         f"(Get-Item '{path}').VersionInfo.ProductVersion"],
                        capture_output=True, text=True, timeout=10
                    )
                    if result.returncode == 0 and result.stdout.strip():
                        version_str = result.stdout.strip()
                        major = int(version_str.split(".")[0])
                        log.info(f"Auto-detected Chrome (exe): {version_str} → major version {major}")
                        return major
                except Exception as e:
                    log.warning(f"Exe Chrome detection failed for {path}: {e}")
        return None

    # ── Linux / VPS: try common Chrome binary names ───────────────────────────
    try:
        for binary in ["google-chrome", "google-chrome-stable", "chromium-browser", "chromium"]:
            result = subprocess.run(
                [binary, "--version"],
                capture_output=True, text=True, timeout=5
            )
            if result.returncode == 0:
                version_str = result.stdout.strip()  # e.g. "Google Chrome 136.0.7103.93"
                major = int(version_str.split()[-1].split(".")[0])
                log.info(f"Auto-detected Chrome: {version_str} → major version {major}")
                return major
    except Exception as e:
        log.warning(f"Could not auto-detect Chrome version: {e}")
    return None   # let UC figure it out itself


def build_driver(download_folder: str, headless: bool):
    # ── On VPS: spin up virtual display so Chrome runs headed ────────────────
    if IS_VPS:
        ensure_virtual_display()

    opts = uc.ChromeOptions()

    prefs = {
        "download.default_directory":         download_folder,
        "download.prompt_for_download":       False,
        "download.directory_upgrade":         True,
        "safebrowsing.enabled":               True,
        "plugins.always_open_pdf_externally": True,
        "intl.accept_languages":              "en-US,en",
    }
    opts.add_experimental_option("prefs", prefs)

    # ── Stability flags (still needed even in headed mode on VPS) ────────────
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-gpu")              # VPS still has no real GPU
    opts.add_argument("--disable-software-rasterizer")
    opts.add_argument("--disable-extensions")
    opts.add_argument("--disable-background-networking")
    opts.add_argument("--no-first-run")
    opts.add_argument("--no-default-browser-check")
    opts.add_argument("--mute-audio")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("--start-maximized")
    opts.add_argument("--lang=en-US,en")

    # ── NO --headless flag at all when using Xvfb ────────────────────────────
    # headless=False both here and in uc.Chrome() call

    # ── Temp profile in shared memory ────────────────────────────────────────
    # shm_available = os.path.exists("/dev/shm") and os.access("/dev/shm", os.W_OK)
    # tmp_base = "/dev/shm" if shm_available else "/tmp"
    # tmp_profile = tempfile.mkdtemp(prefix="chrome_profile_", dir=tmp_base)
    # opts.add_argument(f"--user-data-dir={tmp_profile}")
    # log.info(f"Chrome profile dir: {tmp_profile}")

    chrome_version = get_chrome_version()

    def _start_uc():
        driver_kwargs = {
            "options":  opts,
            "headless": False,   # True headed mode — Xvfb provides the display
        }
        if chrome_version:
            driver_kwargs["version_main"] = chrome_version
        return uc.Chrome(**driver_kwargs)

    try:
        try:
            driver = _start_uc()
            log.info("UC Chrome started in headed mode via Xvfb")
        except Exception as first_err:
            # Most common cause: a stale wrong-version chromedriver cached by UC
            # (the "ChromeDriver only supports Chrome version 150" error). Wipe the
            # cache and retry ONCE with the detected version before giving up on UC.
            log.warning(f"UC Chrome first attempt failed: {first_err}")
            log.info("Clearing UC cache and retrying UC once...")
            clear_uc_cache()
            time.sleep(2)
            driver = _start_uc()
            log.info("UC Chrome started on retry (after cache clear)")

    except Exception as e:
        log.error(f"UC Chrome failed: {e}")
        log.info("Falling back to standard Selenium...")
        # shutil.rmtree(tmp_profile, ignore_errors=True)

        from selenium.webdriver.chrome.service import Service
        from webdriver_manager.chrome import ChromeDriverManager

        std_opts = Options()
        std_opts.add_argument("--no-sandbox")
        std_opts.add_argument("--disable-dev-shm-usage")
        std_opts.add_argument("--disable-gpu")
        std_opts.add_argument("--window-size=1920,1080")
        if IS_VPS and not headless:
            pass   # Xvfb is already running, no --headless needed
        elif headless:
            std_opts.add_argument("--headless=new")
        std_opts.add_experimental_option("prefs", prefs)
        std_opts.add_argument(
            "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/149.0.0.0 Safari/537.36"
        )

        driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()),
            options=std_opts
        )
        log.info("Standard Selenium started")

    # driver._tmp_profile = tmp_profile
    return driver


In [5]:
PROJECT_FOLDER   = os.path.join(os.getcwd(), "sitemaps")
CHILD_FOLDER     = os.path.join(os.getcwd(), "child-sitemaps")
OTHER_CHILD_FOLDER = os.path.join(os.getcwd(), "other-child-sitemaps")
CURR_DIR         = os.path.dirname(os.getcwd())
SITEMAP_INDEX    = "https://www.tiket.com/sitemap/id-id/index.xml.gz"
FOUND_URL_PATH = "found_url.json"

def clean_property_url(url: str) -> str:
    """
    Strip query params and fragment, keep the trailing numeric property ID.
    e.g. https://www.tiket.com/en-id/hotel/indonesia/djineng-rice-terrace-canggu-511001669192830200?room=1&...
    ->   https://www.tiket.com/en-id/hotel/indonesia/djineng-rice-terrace-canggu-511001669192830200
    """
    base = urlunparse(urlparse(url)._replace(query="", fragment=""))
    return base.rstrip("/")

def ask_competitor_id() -> int:
    while True:
        raw = input("competitor_id (integer): ").strip()
        try:
            return int(raw)
        except ValueError:
            print("  -> not an integer, try again.")

def update_found_url(competitor_id: int, cleaned_url: str) -> bool:
    with open(FOUND_URL_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    for entry in data:
        if entry.get("competitor_id") == competitor_id:
            old = entry.get("url", "")
            if old == cleaned_url:
                print(f"[=] competitor_id {competitor_id} already has this URL, nothing to do.")
                return False

            print(f"    name : {entry.get('properties_name')}")
            print(f"    old  : {old}")
            print(f"    new  : {cleaned_url}")
            if input("    write this change? [y/N]: ").strip().lower() != "y":
                print("[-] skipped.")
                return False

            entry["url"] = cleaned_url
            with open(FOUND_URL_PATH, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=4, ensure_ascii=False)
            print("[+] saved.")
            return True

    print(f"[!] competitor_id {competitor_id} not found in found_url.json — nothing written.")
    return False


driver = build_driver(CURR_DIR, headless=IS_VPS)
wait = WebDriverWait(driver, PAGE_TIMEOUT)
driver.get("https://www.tiket.com/en-id/hotel")

try:
    while True:
        print("\n--- Search the property in the browser, then come back here. ---")
        input("Press Enter once the property page is open... ")

        current_url = driver.current_url
        cleaned = clean_property_url(current_url)
        print(f"[url] raw     : {current_url}")
        print(f"[url] cleaned : {cleaned}")

        competitor_id = ask_competitor_id()
        update_found_url(competitor_id, cleaned)

        if input("\nAnother property? [y/N]: ").strip().lower() != "y":
            break
finally:
    driver.quit()



16:53:28 [INFO] Auto-detected Chrome (registry): 150.0.7871.125 → major version 150
16:53:37 [INFO] patching driver executable C:\Users\anton\appdata\roaming\undetected_chromedriver\undetected_chromedriver.exe
16:53:37 [INFO] UC Chrome started in headed mode via Xvfb



--- Search the property in the browser, then come back here. ---
[url] raw     : https://www.tiket.com/en-id/hotel/indonesia/masmara-resort-canggu-311001605066311722?room=1&adult=1&source=search_form_hotel&night=1&id=masmara-resort-canggu-311001605066311722&type=HOTEL&q=MASMARA+Resort+Canggu&checkin=2026-07-17&checkout=2026-07-18&lang=en
[url] cleaned : https://www.tiket.com/en-id/hotel/indonesia/masmara-resort-canggu-311001605066311722
    name : Masmara Resort Canggu
    old  : https://www.tiket.com/en-id/hotel/indonesia/masmara-resort-canggu
    new  : https://www.tiket.com/en-id/hotel/indonesia/masmara-resort-canggu-311001605066311722
[+] saved.

--- Search the property in the browser, then come back here. ---
[url] raw     : https://www.tiket.com/en-id/hotel/indonesia/eastin-ashta-resort-canggu-108001534520245976?room=1&adult=1&source=search_form_hotel&night=1&id=eastin-ashta-resort-canggu-108001534520245976&type=HOTEL&q=Eastin+Ashta+Resort+Canggu&checkin=2026-07-17&checkout=2026

KeyboardInterrupt: Interrupted by user